In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# Paths
BASE_PATH = "/content/drive/MyDrive/research"
INPUT_FILE = os.path.join(BASE_PATH, "dataset1.csv")
ALPHA_BETA_FILE = os.path.join(BASE_PATH, "anca/alpha beta/AlphaBeta_results.csv")
OUTPUT_FILE = os.path.join(BASE_PATH, "dataset2_αβ.csv")

# Load dataset1
df = pd.read_csv(INPUT_FILE)

# Load Alpha_Beta table
ab_df = pd.read_csv(ALPHA_BETA_FILE)

# Ensure File is string
ab_df['File'] = ab_df['File'].astype(str)

# Extract code_module and code_presentation
def extract_module_presentation(fname):
    fname = fname.replace('.xlsx','')
    # remove prefix AlphaBeta_
    if fname.startswith('AlphaBeta_'):
        fname = fname[len('AlphaBeta_'):]
    # Now split remaining into module + presentation
    module = fname[:3]          # first 3 chars, e.g., 'CCC' or 'DDD'
    presentation = fname[3:]    # rest, e.g., '2014B'
    return pd.Series([module, presentation])

ab_df[['code_module','code_presentation']] = ab_df['File'].apply(extract_module_presentation)

# Merge dataset1 with alpha/beta
df = df.merge(
    ab_df[['code_module','code_presentation','Normalized Alpha (%)','Normalized Beta (%)']],
    on=['code_module','code_presentation'],
    how='left'
)

# Only apply alpha/beta where they exist
mask = df['Normalized Alpha (%)'].notna() & df['Normalized Beta (%)'].notna()

# Multiply W1..W9 by alpha fraction
for i in range(1, 10):
    w_col = f'W_{i}'
    if w_col in df.columns:
        df.loc[mask, w_col] = df.loc[mask, w_col] * (df.loc[mask, 'Normalized Alpha (%)']/100)

# Set W10 = Normalized Beta (%)
df.loc[mask, 'W_10'] = df.loc[mask, 'Normalized Beta (%)']

# Set V10 = exam_grade / 100
if 'exam_grade' in df.columns:
    df.loc[mask, 'V_10'] = df.loc[mask, 'exam_grade']/100

# Drop final_grade if exists
if 'final_grade' in df.columns:
    df = df.drop(columns=['final_grade'])

# Drop temporary alpha/beta columns
df = df.drop(columns=['Normalized Alpha (%)','Normalized Beta (%)'])

# Save
df.to_csv(OUTPUT_FILE,index=False)
print(f"Saved dataset2_alpha_beta: {OUTPUT_FILE}")
print("Rows with W_10 != 0:", (df['W_10'] != 0).sum())

Saved dataset2_alpha_beta: /content/drive/MyDrive/research/dataset2_αβ.csv
Rows with W_10 != 0: 8959


In [ ]:
import pandas as pd
import os

# Paths
BASE_PATH = "/content/drive/MyDrive/research"
DATASET2_FILE = os.path.join(BASE_PATH, "dataset2_αβ.csv")

# Load dataset2_alpha_beta
df2 = pd.read_csv(DATASET2_FILE)

# Find rows where W_10 is non-zero
nonzero_rows = df2.index[df2['W_10'] != 0]

# Count of non-zero W_10
w10_nonzero_count = len(nonzero_rows)
print(f"Number of rows with W_10 != 0: {w10_nonzero_count}")

# First row index with W_10 != 0
if w10_nonzero_count > 0:
    first_row = nonzero_rows[0]
    print(f"First row with W_10 != 0 is at index: {first_row}")
else:
    print("No rows have W_10 != 0")

Number of rows with W_10 != 0: 8959
First row with W_10 != 0 is at index: 6865
